In [5]:
## Imports from 0_pull_format_all_api_data
import pandas as pd
import numpy as np
import re
import requests
import yaml
import os

## Imports for wikipedia pulls
import pywikibot as pw
import time
pw.config.user_agent = 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu) Pywikibot/11.6.0'

## Imports for test visualization, if needed
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit

## Imports for google search trends
from pytrends.request import TrendReq

## Set font parameters, if needed
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]


## Repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [15]:
## Read all relevant datasets

## Part 0
basic_info_df = pd.read_csv('../data/basic_info.csv')

## Part 1
top_articles_df = pd.read_csv('../data/top_articles_list.csv')
daily_wikipedia_df = pd.read_csv('../data/daily_wikipedia_views_all.csv')
country_daily_df = pd.read_csv('../data/country_daily_views.csv')

## Part 2
diaspora_2022_df = pd.read_csv('../data/diaspora_2022.csv')
diaspora_2024_df = pd.read_csv('../data/diaspora_2024.csv')
regional_search_interest_df = pd.read_csv('../data/regional_search_interest.csv')

## Part 3
tweet_data_2022_df = pd.read_csv('../data/fifa_world_cup_2022_tweets.csv')

In [48]:
## Part 1: Need to concatenate country_daily_df and daily_wikipedia_df

country_daily_df["subcategory_title_fixed"] = country_daily_df["article_title"]

merged_daily_df = pd.concat([daily_wikipedia_df, country_daily_df], axis=0, ignore_index=True)

merged_daily_df['date'] = pd.to_datetime(merged_daily_df['date'], format='%Y%m%d')
merged_daily_df['year played'] = merged_daily_df['date'].dt.year

merged_daily_df

daily_wikipedia_df.shape

,country,subcategory_title_fixed,article_title,date,views,year played
0,Australia,Culture of Australia,Culture_of_Australia,2026-06-01,354,2026.0
1,Australia,Culture of Australia,Culture_of_Australia,2026-06-02,341,2026.0
2,Australia,Culture of Australia,Culture_of_Australia,2026-06-03,427,2026.0
3,Australia,Culture of Australia,Culture_of_Australia,2026-06-04,371,2026.0
4,Australia,Culture of Australia,Culture_of_Australia,2026-06-05,331,2026.0
...,...,...,...,...,...,...
88625,Wales,Wales,Wales,2022-12-27,6694,2022.0
88626,Wales,Wales,Wales,2022-12-28,6289,2022.0
88627,Wales,Wales,Wales,2022-12-29,6492,2022.0
88628,Wales,Wales,Wales,2022-12-30,6167,2022.0


(88082, 5)

In [50]:
## Now apply a left join with basic_info to condense all match info into one dataframe

total_wikipedia_df = pd.merge(merged_daily_df, basic_info_df, how="left", on=["country", "year played"])
total_wikipedia_df

## Save data for future use in 4_predictive_engagement_model.ipynb
total_wikipedia_df.to_csv('../data/total_wikipedia_dataset.csv', index=False)

,country,subcategory_title_fixed,article_title,date,views,year played,match date,outcome,match type,score
0,Australia,Culture of Australia,Culture_of_Australia,2026-06-01,354,2026.0,20260619.0,win,group,2-0
1,Australia,Culture of Australia,Culture_of_Australia,2026-06-02,341,2026.0,20260619.0,win,group,2-0
2,Australia,Culture of Australia,Culture_of_Australia,2026-06-03,427,2026.0,20260619.0,win,group,2-0
3,Australia,Culture of Australia,Culture_of_Australia,2026-06-04,371,2026.0,20260619.0,win,group,2-0
4,Australia,Culture of Australia,Culture_of_Australia,2026-06-05,331,2026.0,20260619.0,win,group,2-0
...,...,...,...,...,...,...,...,...,...,...
88625,Wales,Wales,Wales,2022-12-27,6694,2022.0,20221121.0,draw,group,1-1
88626,Wales,Wales,Wales,2022-12-28,6289,2022.0,20221121.0,draw,group,1-1
88627,Wales,Wales,Wales,2022-12-29,6492,2022.0,20221121.0,draw,group,1-1
88628,Wales,Wales,Wales,2022-12-30,6167,2022.0,20221121.0,draw,group,1-1


In [56]:
## Now filter out most popular articles and apply conditional statement to keep popular articles and country main page articles in merged_daily_df

top_articles_df

cleaned_wikipedia_df = merged_daily_df[(merged_daily_df["article_title"].isin(top_articles_df["article_title"])) | 
    (merged_daily_df["article_title"] == merged_daily_df["country"])]

cleaned_wikipedia_df = cleaned_wikipedia_df.reset_index(drop=True)
cleaned_wikipedia_df

cleaned_wikipedia_df.to_csv('../data/cleaned_wikipedia_data.csv', index=False)

,country,subcategory_title_fixed,article_title,mean_views
0,England,Culture of England,The_Beatles,16766.098361
1,Netherlands,Culture of the Netherlands,Sinterklaas,4660.983607
2,Iran,Culture of Iran,Persian_language,4656.245902
3,Wales,Welsh language,Welsh_language,3820.704918
4,Australia,Culture of Australia,Bluey_(TV_series),2951.150000
5,Wales,Culture of Wales,Mari_Lwyd,2132.213115
6,England,Politics of England,Counties_of_England,1808.131148
7,Iran,Politics of Iran,Supreme_Leader_of_Iran,1721.081967
8,Belgium,Culture of Belgium,The_Adventures_of_Tintin,1652.333333
9,Iran,Economy of Iran,Economy_of_Iran,1141.163934


,country,subcategory_title_fixed,article_title,date,views,year played
0,Australia,Culture of Australia,Bluey_(TV_series),2026-06-01,2927,2026.0
1,Australia,Culture of Australia,Bluey_(TV_series),2026-06-02,2876,2026.0
2,Australia,Culture of Australia,Bluey_(TV_series),2026-06-03,2768,2026.0
3,Australia,Culture of Australia,Bluey_(TV_series),2026-06-04,2953,2026.0
4,Australia,Culture of Australia,Bluey_(TV_series),2026-06-05,2878,2026.0
...,...,...,...,...,...,...
2974,Wales,Wales,Wales,2022-12-27,6694,2022.0
2975,Wales,Wales,Wales,2022-12-28,6289,2022.0
2976,Wales,Wales,Wales,2022-12-29,6492,2022.0
2977,Wales,Wales,Wales,2022-12-30,6167,2022.0
